# Tessera-PQC Tutorial 1: Post-Quantum Cryptography Basics

This notebook introduces post-quantum cryptographic algorithms implemented in Tessera-PQC:
- **Kyber**: Key Encapsulation Mechanism (KEM) for secure key exchange
- **Dilithium**: Digital signature scheme

Both are NIST PQC standards based on lattice problems.

In [ ]:
import tessera as t
import numpy as np

print(f"Tessera-PQC version: {t.__version__}")

## 1. Kyber Key Encapsulation Mechanism (KEM)

Kyber is used for secure key exchange. The workflow is:
1. **KeyGen**: Generate public/private key pair
2. **Encapsulation**: Sender creates ciphertext and shared secret from public key
3. **Decapsulation**: Receiver extracts same shared secret using private key

In [ ]:
# Generate Kyber-768 key pair (NIST security level 3)
public_key, secret_key = t.kyber_keygen("768")

print(f"Public key size:  {len(public_key):,} bytes")
print(f"Secret key size:  {len(secret_key):,} bytes")

In [ ]:
# Encapsulation (sender side)
ciphertext, shared_secret_enc = t.kyber_encaps(public_key, "768")

print(f"Ciphertext size:  {len(ciphertext):,} bytes")
print(f"Shared secret:    {shared_secret_enc[:16].hex()}...")

In [ ]:
# Decapsulation (receiver side)
shared_secret_dec = t.kyber_decaps(secret_key, ciphertext, "768")

print(f"Shared secret:    {shared_secret_dec[:16].hex()}...")
print(f"\nSecrets match: {shared_secret_enc == shared_secret_dec}")

### Kyber Security Levels

| Variant | NIST Level | Security | Public Key | Ciphertext |
|---------|------------|----------|------------|------------|
| Kyber-512 | 1 | ~AES-128 | 800 bytes | 768 bytes |
| Kyber-768 | 3 | ~AES-192 | 1,184 bytes | 1,088 bytes |
| Kyber-1024 | 5 | ~AES-256 | 1,568 bytes | 1,568 bytes |

In [ ]:
# Compare all Kyber variants
for variant in ["512", "768", "1024"]:
    pk, sk = t.kyber_keygen(variant)
    ct, ss = t.kyber_encaps(pk, variant)
    print(f"Kyber-{variant}: pk={len(pk):,} bytes, ct={len(ct):,} bytes, ss={len(ss)} bytes")

## 2. Dilithium Digital Signatures

Dilithium is used for digital signatures. The workflow is:
1. **KeyGen**: Generate public/private key pair
2. **Sign**: Create signature for a message using private key
3. **Verify**: Verify signature using public key

In [ ]:
# Generate Dilithium3 key pair (NIST security level 3)
public_key, secret_key = t.dilithium_keygen("3")

print(f"Public key size:  {len(public_key):,} bytes")
print(f"Secret key size:  {len(secret_key):,} bytes")

In [ ]:
# Sign a message
message = b"Hello, post-quantum world!"

signature = t.dilithium_sign(secret_key, message, "3")

print(f"Message:         '{message.decode()}'")
print(f"Signature size:  {len(signature):,} bytes")

In [ ]:
# Verify the signature
is_valid = t.dilithium_verify(public_key, message, signature, "3")

print(f"Signature valid: {is_valid}")

# Try verifying a modified message
modified_message = b"Hello, post-quantum world?"
is_valid_modified = t.dilithium_verify(public_key, modified_message, signature, "3")
print(f"Modified message valid: {is_valid_modified}")

### Dilithium Security Levels

| Variant | NIST Level | Security | Public Key | Signature |
|---------|------------|----------|------------|------------|
| Dilithium2 | 2 | ~AES-128 | 1,312 bytes | 2,420 bytes |
| Dilithium3 | 3 | ~AES-192 | 1,952 bytes | 3,293 bytes |
| Dilithium5 | 5 | ~AES-256 | 2,592 bytes | 4,595 bytes |

In [ ]:
# Compare all Dilithium variants
msg = b"Test message"
for variant in ["2", "3", "5"]:
    pk, sk = t.dilithium_keygen(variant)
    sig = t.dilithium_sign(sk, msg, variant)
    print(f"Dilithium{variant}: pk={len(pk):,} bytes, sk={len(sk):,} bytes, sig={len(sig):,} bytes")

## 3. Performance Benchmarks

Let's measure the performance of these algorithms.

In [ ]:
# Benchmark Kyber
print("Kyber Performance (10 iterations):")
print("-" * 50)
for variant in ["512", "768", "1024"]:
    result = t.benchmark_kyber(variant, iterations=10)
    print(f"Kyber-{variant}:")
    print(f"  KeyGen: {result['keygen_ms']:.2f} ms")
    print(f"  Encaps: {result['encaps_ms']:.2f} ms")
    print(f"  Decaps: {result['decaps_ms']:.2f} ms")

In [ ]:
# Benchmark Dilithium
print("Dilithium Performance (5 iterations):")
print("-" * 50)
for variant in ["2", "3", "5"]:
    result = t.benchmark_dilithium(variant, iterations=5)
    print(f"Dilithium{variant}:")
    print(f"  KeyGen: {result['keygen_ms']:.2f} ms")
    print(f"  Sign:   {result['sign_ms']:.2f} ms")
    print(f"  Verify: {result['verify_ms']:.2f} ms")

## 4. Understanding NTT (Number Theoretic Transform)

Both Kyber and Dilithium use NTT for efficient polynomial multiplication.

In [ ]:
from tessera.core.math_fast import ntt_kyber, intt_kyber

# Create a random polynomial
poly = np.random.randint(0, 3329, 256, dtype=np.int64)

# Apply NTT
poly_ntt = ntt_kyber(poly.copy())

# Apply inverse NTT
poly_recovered = intt_kyber(poly_ntt.copy())

print(f"Original:   {poly[:8]}...")
print(f"After INTT: {poly_recovered[:8]}...")
print(f"Round-trip correct: {np.array_equal(poly, poly_recovered)}")

## Summary

In this notebook, you learned:

1. **Kyber KEM**: Secure key exchange using lattice-based cryptography
   - Generate keys, encapsulate, and decapsulate to derive shared secrets
   - Three security levels (512, 768, 1024)

2. **Dilithium Signatures**: Digital signatures using lattice-based cryptography
   - Generate keys, sign messages, and verify signatures
   - Three security levels (2, 3, 5)

3. **NTT**: The core mathematical operation enabling efficient polynomial arithmetic

**Next**: See Tutorial 2 for side-channel analysis techniques.